In [26]:
# mount google drive and set up paths
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
# %%writefile setup.py
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

torch.set_grad_enabled(False)

# ── Path constants ─────────────────────────────────────────────────────────────
DRIVE_BASE = Path('/content/drive/MyDrive')

SAM_REPO = DRIVE_BASE / 'segment_anything_2'
SAM_CHECKPOINT = SAM_REPO / 'checkpoints'


LIGHTGLUE_REPO = DRIVE_BASE / 'LightGlue'

SWINIR_REPO = DRIVE_BASE / 'SwinIR'
SWINIR_CHECKPOINT = SWINIR_REPO / 'checkpoints'

INPUT_DIR = DRIVE_BASE / 'input'

OUTPUT_DIR       = DRIVE_BASE / 'output'

SAM_BASE_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824"
SWINIR_BASE_URL = 'https://github.com/JingyunLiang/SwinIR/releases/download/v0.0'


sam2_checkpoints = {
    "tiny": f"sam2.1_hiera_tiny.pt",
    "small": f"sam2.1_hiera_small.pt",
    "large": f"sam2.1_hiera_large.pt",
}

swinir_checkpoints = {
    "real_sr": "003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth",
    "lightweight": "002_lightweightSR_DIV2K_s64w8_SwinIR-S_x4.pth",
}

sam2_cfgs = {
    "tiny": "configs/sam2.1/sam2.1_hiera_t.yaml",
    "small": "configs/sam2.1/sam2.1_hiera_s.yaml",
    "large": "configs/sam2.1/sam2.1_hiera_l.yaml"
}


# ── Repo setup ─────────────────────────────────────────────────────────────────
def _run(cmd: str) -> bool:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] command failed: {cmd!r}\n{result.stderr[:300]}")
    return result.returncode == 0

def download_file(url, dest_path):
    dest_path = Path(dest_path) # make into path if not already
    print(f"Downloading {dest_path.name}...")
    try:
        cmd = f'wget --user-agent="Mozilla/5.0" -O "{dest_path}" "{url}"'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(f"Downloaded: {dest_path.name}")
        return True
    except Exception as e:
        print(f"Failed to download {dest_path.name}: {e}")
        return False

def check_checkpoints(model_name, checkpoint_base_path, checkpoint_base_url):
    checkpoint_path = Path(f"{checkpoint_base_path}/{model_name}")
    checkpoint_url = f"{checkpoint_base_url}/{model_name}"

    # Check if file exists AND is not empty
    if checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:
        size_mb = checkpoint_path.stat().st_size / (1024 * 1024)
        print(f"{model_name} already exists ({size_mb:.1f} MB)")
    else:
        # Remove empty file if it exists
        if checkpoint_path.exists():
            checkpoint_path.unlink()

        print(f"{model_name} missing, downloading...")
        success = download_file(checkpoint_url, checkpoint_path)

        if success and checkpoint_path.stat().st_size > 0:
            size_mb = checkpoint_path.stat().st_size / (1024 * 1024)
            print(f"  Size: {size_mb:.1f} MB")
        else:
            print(f"  Download failed or file is empty")
            if checkpoint_path.exists():
                checkpoint_path.unlink()  # Clean up failed download

def download_repos_and_setup() -> None:
    if not SAM_REPO.exists():
        print("Cloning segment-anything-2 (sam2)...")
        _run(f"git clone https://github.com/facebookresearch/sam2.git {SAM_REPO}")
    else:
        print(f"SAM2 repo already exists at {SAM_REPO}")

    # Install sam2 package
    print("Installing sam2 package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(SAM_REPO)])

    # Add repo to path so imports work

    if str(SAM_REPO) not in sys.path:
        sys.path.insert(0, str(SAM_REPO))

    # --- 2. Check and download SAM 2.1 checkpoints ---
    SAM_CHECKPOINT.mkdir(parents=True, exist_ok=True)

    # Check each checkpoint and download if missing
    print(f"\nChecking sam2 checkpoints in: {SAM_CHECKPOINT}")
    for model_name, model_type in sam2_checkpoints.items():
        check_checkpoints(model_type, SAM_CHECKPOINT, SAM_BASE_URL)

    print("\nAll sam2 checkpoints verified!")

    # LightGlue
    if not LIGHTGLUE_REPO.exists():
        print("Cloning LightGlue …")
        _run(f"git clone https://github.com/cvg/LightGlue.git {LIGHTGLUE_REPO}")

    if str(LIGHTGLUE_REPO) not in sys.path:
        sys.path.insert(0, str(LIGHTGLUE_REPO))

    print("\nInstalling LightGlue package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(LIGHTGLUE_REPO)])

    # SwinIR – no setup.py, just add to sys.path
    if not SWINIR_REPO.exists():
        print("\nCloning SwinIR …")
        _run(f"git clone https://github.com/JingyunLiang/SwinIR.git {SWINIR_REPO}")

    SWINIR_CHECKPOINT.mkdir(parents=True, exist_ok=True)
    print("SwinIR – no setup.py, just adding to sys.path...")
    if str(SWINIR_REPO) not in sys.path:
        sys.path.insert(0, str(SWINIR_REPO))

    # Check each checkpoint and download if missing
    print(f"\nChecking swinir checkpoints in: {SWINIR_CHECKPOINT}")
    for model_type, model_name in swinir_checkpoints.items():
        check_checkpoints(model_name, SWINIR_CHECKPOINT, SWINIR_BASE_URL)

    print("\nAll swinir checkpoints verified!")

    print("\nAll repos ready.")


# ── Model loaders ──────────────────────────────────────────────────────────────
def load_swinir_model(model_type: str = "real_sr"):
    """Load the lightweight or real_sr SwinIR x4 SR model."""
    import torch
    from models.network_swinir import SwinIR as net  # requires SWINIR_REPO on sys.path

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    assert model_type in ("real_sr", "lightweight"), "only real_sr and lightweight model_type are allowed"
    if model_type == "lightweight":
        model = net(
            upscale=4,
            in_chans=3,
            img_size=64,
            window_size=8,
            img_range=1.0,
            depths=[6, 6, 6, 6],
            embed_dim=60,
            num_heads=[6, 6, 6, 6],
            mlp_ratio=2,
            upsampler="pixelshuffledirect",
            resi_connection="1conv",
        )
    elif model_type == "real_sr":
        # these setting are the bigger SR model
        model = net(
            upscale=4,
            in_chans=3,
            img_size=64,
            window_size=8,
            img_range=1.0,
            depths=[6,6,6,6,6,6,6,6,6],
            embed_dim=240,
            num_heads=[8,8,8,8,8,8,8,8,8],
            mlp_ratio=2,
            upsampler='nearest+conv',
            resi_connection='3conv'
        )

    checkpoint = SWINIR_CHECKPOINT / swinir_checkpoints[model_type]
    pretrained = torch.load(checkpoint, map_location="cpu")
    key = "params_ema" if "params_ema" in pretrained else "params"
    model.load_state_dict(pretrained[key], strict=True)
    model.eval().to(device)

    print(f"SwinIR loaded from {checkpoint.name}")
    return model

def load_sam2_model(model_type: str = "small", device: str = None):
    """Load SAM 2 Model by type - large, small, tiny and return (model, mask_generator)"""
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    if device is None:
        device = 'cuda' if torch.cuda.is_available() else "cpu"

    SAM_CHECKPOINT = SAM_REPO / 'checkpoints'
    model_checkpoint = sam2_checkpoints[model_type]
    model = f"{SAM_CHECKPOINT}/{model_checkpoint}"
    model_cfg = sam2_cfgs[model_type]
    sam2_model = build_sam2(model_cfg, model, device=device)

    predictor = SAM2ImagePredictor(sam2_model)

    print(f"SAM2 loaded from {model_checkpoint}")

    return sam2_model, predictor

def load_lightglue_models(filter_threshold: float = 0.05, depth_confidence=-1, width_confidence=-1):
    """Load SuperPoint + LightGlue from local checkpoints."""
    import torch
    from lightglue import LightGlue, SuperPoint

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # SuperPoint with local weights
    extractor = SuperPoint(max_num_keypoints=1024).eval().to(device)

    # LightGlue with local weights
    matcher = LightGlue(
        features="superpoint",
        depth_confidence=-1,
        width_confidence=-1,
        filter_threshold=filter_threshold,
    ).eval().to(device)

    print("SuperPoint + LightGlue loaded from local checkpoints")
    return extractor, matcher

Writing setup.py


In [28]:
download_repos_and_setup()

SAM2 repo already exists at /content/drive/MyDrive/segment_anything_2
Installing sam2 package...

Checking sam2 checkpoints in: /content/drive/MyDrive/segment_anything_2/checkpoints
sam2.1_hiera_tiny.pt already exists (148.8 MB)
sam2.1_hiera_small.pt already exists (175.9 MB)
sam2.1_hiera_large.pt already exists (856.5 MB)

All sam2 checkpoints verified!

Installing LightGlue package...
SwinIR – no setup.py, just adding to sys.path...

Checking swinir checkpoints in: /content/drive/MyDrive/SwinIR/checkpoints
003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth already exists (135.9 MB)
002_lightweightSR_DIV2K_s64w8_SwinIR-S_x4.pth already exists (16.4 MB)

All swinir checkpoints verified!

All repos ready.


In [66]:
%%writefile app.py
import streamlit as st
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch
from streamlit_image_coordinates import streamlit_image_coordinates
from setup import load_lightglue_models, load_sam2_model
from lightglue.utils import rbd
from lightglue import viz2d

st.set_page_config(layout="wide")
st.title("SAM2 + LightGlue")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DISPLAY_WIDTH = 640

@st.cache_resource
def load_all_tools():
    extractor, matcher = load_lightglue_models()
    sam2_model, predictor = load_sam2_model()
    return extractor, matcher, predictor

extractor, matcher, predictor = load_all_tools()

for i in ["1", "2"]:
    for key in ["points", "labels", "mask", "img_pil", "last_click"]:
        st.session_state.setdefault(f"{key}_{i}", [] if "points" in key or "labels" in key else None)

def create_tensor_from_mask(mask):
    # Mask is 2D bool/uint8, expand to 3-channel float tensor
    if mask.ndim == 2:
        mask = np.stack([mask, mask, mask], axis=-1)
    # HWC -> CHW
    image = np.transpose(mask, (2, 0, 1))
    return torch.from_numpy(image).float().unsqueeze(0).to(DEVICE)

def get_keypoint_matches(image0, image1, conf_thresh):
    feats0 = extractor.extract(image0)
    feats1 = extractor.extract(image1)

    # Update threshold on existing matcher
    matcher.conf.filter_threshold = conf_thresh

    matches01 = matcher({
        "image0": feats0,
        "image1": feats1,
    })

    feats0, feats1, matches01 = [rbd(x) for x in [feats0, feats1, matches01]]
    kpts0, kpts1, matches = feats0["keypoints"], feats1["keypoints"], matches01["matches"]
    return kpts0, kpts1, kpts0[matches[..., 0]], kpts1[matches[..., 1]], matches01

def draw_points(base_img, img_id):
    img = base_img.convert("RGB")
    draw = ImageDraw.Draw(img)
    for (dx, dy), lbl in zip(st.session_state[f"points_{img_id}"], st.session_state[f"labels_{img_id}"]):
        color = (0, 220, 0) if lbl == 1 else (220, 0, 0)
        draw.ellipse([(dx-8, dy-8), (dx+8, dy+8)], fill=color, outline="white", width=2)
    return img

def run_sam_logic(img_id):
    if not st.session_state[f"points_{img_id}"]:
        st.warning("No points")
        return

    img_pil = st.session_state[f"img_pil_{img_id}"]
    img_np = np.array(img_pil)

    # Handle grayscale
    if img_np.ndim == 2:
        img_np = np.stack([img_np, img_np, img_np], axis=-1)

    orig_w, orig_h = img_pil.size
    display_h = int(orig_h * (DISPLAY_WIDTH / orig_w))
    scale_x = orig_w / DISPLAY_WIDTH
    scale_y = orig_h / display_h

    predictor.set_image(img_np)

    # Scale display coords to original image coords
    coords = np.array([
        [int(x * scale_x), int(y * scale_y)]
        for x, y in st.session_state[f"points_{img_id}"]
    ])
    labs = np.array(st.session_state[f"labels_{img_id}"])

    with torch.inference_mode():
        masks, scores, _ = predictor.predict(
            point_coords=coords,
            point_labels=labs,
            multimask_output=True
        )
    st.session_state[f"mask_{img_id}"] = masks[np.argmax(scores)]

t1, t2, t3 = st.tabs(["Image 1", "Image 2", "Match"])

for i in ["1", "2"]:
    with (t1 if i == "1" else t2):
        up = st.file_uploader(f"Img {i}", type=["jpg", "png"], key=f"up_{i}")
        if up:
            # Force RGB on load
            img = Image.open(up)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            st.session_state[f"img_pil_{i}"] = img

            w, h = img.size
            disp_h = int(h * (DISPLAY_WIDTH / w))
            disp = img.resize((DISPLAY_WIDTH, disp_h))

            c1, c2 = st.columns([1, 3])
            with c1:
                mode = st.radio(f"Mode {i}", ["Add", "Remove", "Delete"], key=f"m_{i}")
                if st.button(f"Segment {i}", key=f"s_{i}"):
                    run_sam_logic(i)
                    st.rerun()

                if st.button(f"Undo {i}", key=f"u_{i}") and st.session_state[f"points_{i}"]:
                    st.session_state[f"points_{i}"].pop()
                    st.session_state[f"labels_{i}"].pop()
                    st.rerun()

                if st.button(f"Clear {i}", key=f"c_{i}"):
                    st.session_state[f"points_{i}"] = []
                    st.session_state[f"labels_{i}"] = []
                    st.session_state[f"mask_{i}"] = None
                    st.rerun()

            with c2:
                click = streamlit_image_coordinates(
                    draw_points(disp, i),
                    key=f"cl_{i}",
                    width=DISPLAY_WIDTH
                )

                if click and click != st.session_state[f"last_click_{i}"]:
                    st.session_state[f"last_click_{i}"] = click
                    cx, cy = click["x"], click["y"]

                    if mode == "Delete" and st.session_state[f"points_{i}"]:
                        dists = [(cx-x)**2 + (cy-y)**2 for x, y in st.session_state[f"points_{i}"]]
                        idx = np.argmin(dists)
                        st.session_state[f"points_{i}"].pop(idx)
                        st.session_state[f"labels_{i}"].pop(idx)
                        st.rerun()
                    else:
                        st.session_state[f"points_{i}"].append([cx, cy])
                        st.session_state[f"labels_{i}"].append(1 if mode == "Add" else 0)
                        st.rerun()

                if st.session_state[f"mask_{i}"] is not None:
                    st.image(st.session_state[f"mask_{i}"], clamp=True, caption="Mask")

# Add after existing imports at top
import cv2

# Add these functions before the Streamlit code

def estimate_transform(kpts0, kpts1):
    """Estimate affine transformation from kpts0 to kpts1"""
    M, inliers = cv2.estimateAffinePartial2D(
        kpts0.astype(np.float32),
        kpts1.astype(np.float32),
        method=cv2.RANSAC,
        ransacReprojThreshold=3.0
    )

    if M is None:
        return None, None, 0

    # Extract scale from transformation matrix
    scale = np.sqrt(M[0,0]**2 + M[0,1]**2)

    return M, inliers, scale

def apply_transform_overlay(img_source, img_target, M, alpha=0.5):
    """Warp source image and overlay on target with red/blue coloring"""
    h, w = img_target.shape[:2]

    # Warp source to match target
    warped = cv2.warpAffine(img_source, M, (w, h), flags=cv2.INTER_LINEAR)

    # Convert grayscale to RGB with different colors
    # Target = Red channel
    # Warped = Blue channel
    overlay = np.zeros((h, w, 3), dtype=np.uint8)

    # Normalize if needed
    if img_target.max() > 0:
        target_norm = (img_target.astype(float) / img_target.max() * 255).astype(np.uint8)
    else:
        target_norm = img_target

    if warped.max() > 0:
        warped_norm = (warped.astype(float) / warped.max() * 255).astype(np.uint8)
    else:
        warped_norm = warped

    overlay[:, :, 0] = warped_norm  # Blue channel = warped source
    overlay[:, :, 2] = target_norm  # Red channel = target

    # Where they overlap = purple/magenta

    return overlay, warped

# Update the Match tab
with t3:
    if st.session_state.mask_1 is not None and st.session_state.mask_2 is not None:
        thresh = st.slider("Filter Threshold", 0.0, 0.2, 0.05, 0.01)

        col1, col2 = st.columns(2)
        with col1:
            run_match = st.button("Match Keypoints")
        with col2:
            run_transform = st.button("Estimate Transform")

        if run_match:
            t0 = create_tensor_from_mask(st.session_state.mask_1)
            t1t = create_tensor_from_mask(st.session_state.mask_2)

            _, _, mk0, mk1, m01 = get_keypoint_matches(t0, t1t, thresh)

            # Store matches in session state
            st.session_state['matched_kpts_1'] = mk0.cpu().numpy()
            st.session_state['matched_kpts_2'] = mk1.cpu().numpy()

            st.write(f"Found {len(mk0)} matches")

            # Store the match figure
            plt.close('all')
            viz2d.plot_images([t0[0][0].cpu(), t1t[0][0].cpu()])
            viz2d.plot_matches(mk0.cpu(), mk1.cpu(), color="lime", lw=0.2)
            st.session_state['match_fig'] = plt.gcf()

        # Always show matches if they exist
        if 'match_fig' in st.session_state:
            st.subheader("Keypoint Matches")
            st.pyplot(st.session_state['match_fig'])

        if run_transform and 'matched_kpts_1' in st.session_state:
            kpts1 = st.session_state['matched_kpts_1']
            kpts2 = st.session_state['matched_kpts_2']

            M, inliers, scale = estimate_transform(kpts1, kpts2)

            if M is None:
                st.error("Failed to estimate transform")
            else:
                st.success(f"Transform estimated with {np.sum(inliers)} inliers")
                st.write(f"**Scale factor:** {scale:.5f}")
                st.write("**Transformation Matrix:**")
                st.code(f"{M}")

                # Store transform
                st.session_state['transform_M'] = M
                st.session_state['transform_scale'] = scale

        # Always show transform overlay if it exists
        if 'transform_M' in st.session_state:
            st.subheader("Aligned Overlay")

            M = st.session_state['transform_M']
            scale = st.session_state['transform_scale']

            # Get original images (not masks)
            img1 = np.array(st.session_state.img_pil_1.convert('L'))
            img2 = np.array(st.session_state.img_pil_2.convert('L'))

            # Apply transform and create overlay
            alpha_blend = st.slider("Overlay transparency", 0.0, 1.0, 0.5, 0.05)
            overlay, warped = apply_transform_overlay(img1, img2, M, alpha=alpha_blend)

            # Display results
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            axes[0].imshow(img2)
            axes[0].set_title("Target (Image 2)")
            axes[0].axis('off')

            axes[1].imshow(warped)
            axes[1].set_title(f"Warped Source (scale={scale:.4f})")
            axes[1].axis('off')

            axes[2].imshow(overlay)
            axes[2].set_title("Overlay (Red=Target, Blue=Source)")
            axes[2].axis('off')

            plt.tight_layout()
            st.pyplot(fig)
            plt.close()
    else:
        st.info("Need both masks")

Overwriting app.py


In [45]:
%%capture
!pip install streamlit pyngrok streamlit-image-coordinates -q

In [60]:
import subprocess
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)

CompletedProcess(args=['pkill', '-f', 'ngrok'], returncode=0, stdout=b'', stderr=b'')

In [61]:
import subprocess, time
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_AUTH"))

# Write logs to files so we can read them
with open("/content/streamlit.log", "w") as log:
    proc = subprocess.Popen(
        ["streamlit", "run", "/content/app.py", "--server.port", "8501", "--server.headless", "true"],
        stdout=log,
        stderr=log
    )

time.sleep(5)
tunnel = ngrok.connect(8501)
print(f"URL: {tunnel.public_url}")

URL: https://eloquent-lowell-valiantly.ngrok-free.dev
